In [10]:
# Preliminares 

import re
import pandas as pd

In [11]:
# Datos 

ubicaciones_clientes = pd.read_excel("../Limpia/Ubicaciones_direcciones.xlsx")
tareas = pd.read_excel("../Limpia/Tareas-limpio.xlsx")

In [12]:
tareas.info()
tareas.sample(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9998 entries, 0 to 9997
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PROYECTO      1750 non-null   object 
 1   TAREACLIENTE  1733 non-null   object 
 2   PERIODICIDAD  1461 non-null   object 
 3   ORDEN         1679 non-null   float64
 4   TIPO          1750 non-null   object 
 5   CODCLI2       9967 non-null   float64
dtypes: float64(2), object(4)
memory usage: 468.8+ KB


,PROYECTO,TAREACLIENTE,PERIODICIDAD,ORDEN,TIPO,CODCLI2
769,Ruta Hugo Lunes,"11960, CHURRERIA NAGUS (10),Parque Norte",cada lunes,16.0,Ruta,11960.0
7906,NaN,NaN,NaN,NaN,NaN,0.0


In [13]:
# check 
ubicaciones_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CODCLI2    738 non-null    int64         
 1   NOMCLI     737 non-null    object        
 2   LATITUD    738 non-null    float64       
 3   LONGITUD   738 non-null    float64       
 4   UBICACIÓN  738 non-null    object        
 5   FECHA      738 non-null    datetime64[ns]
 6   METODO     738 non-null    object        
 7   CONFIANZA  738 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(2), object(3)
memory usage: 46.2+ KB


In [14]:
# Sacamos Fecha

ubicaciones_clientes =ubicaciones_clientes.drop(columns=["FECHA"])
ubicaciones_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CODCLI2    738 non-null    int64  
 1   NOMCLI     737 non-null    object 
 2   LATITUD    738 non-null    float64
 3   LONGITUD   738 non-null    float64
 4   UBICACIÓN  738 non-null    object 
 5   METODO     738 non-null    object 
 6   CONFIANZA  738 non-null    int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 40.5+ KB


In [15]:
ubicaciones_clientes.head(5)

,CODCLI2,NOMCLI,LATITUD,LONGITUD,UBICACIÓN,METODO,CONFIANZA
0,82122,NUEVOS RUMBOS SALINAS,-34.765666,-55.837849,"http://maps.google.com/?q=-34.7656659,-55.8378493",google_q_param,90
1,81125,LA ESQUINA jacqueline vanesca (cobranza),-34.780645,-55.838592,"http://maps.google.com/?q=-34.78064530920805,-...",google_q_param,90
2,11941,"11941, LO DE RAMON",-34.766682,-55.742948,"http://maps.google.com/?q=-34.76668227159642,-...",google_q_param,90
3,80318,"80318,MINIMARKET LA 10",-34.761506,-55.747363,"http://maps.google.com/?q=-34.76150608583312,-...",google_q_param,90
4,80324,"80324,KIOSCO FANTASIA",-34.774175,-55.762672,"http://maps.google.com/?q=-34.774175363604215,...",google_q_param,90


In [16]:
tareas['CODCLI2'] = tareas['CODCLI2'].astype('Int64')

In [17]:
tareas.head(5)

,PROYECTO,TAREACLIENTE,PERIODICIDAD,ORDEN,TIPO,CODCLI2
0,Mantenimiento Rutas,NaN,NaN,NaN,Mant,<NA>
1,PreRuta,Walter,NaN,1.0,PreR,<NA>
2,Rutas Mauricio,NaN,NaN,NaN,Ruta,<NA>
3,Ruta Martin Lunes,NaN,NaN,NaN,Ruta,<NA>
4,Ruta Martin Martes,NaN,NaN,NaN,Ruta,<NA>


In [18]:
tareas = tareas.drop(columns=['TAREACLIENTE'])

In [19]:
tareas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9998 entries, 0 to 9997
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PROYECTO      1750 non-null   object 
 1   PERIODICIDAD  1461 non-null   object 
 2   ORDEN         1679 non-null   float64
 3   TIPO          1750 non-null   object 
 4   CODCLI2       9967 non-null   Int64  
dtypes: Int64(1), float64(1), object(3)
memory usage: 400.4+ KB


In [20]:
# Hay columnas repetidas tanto en codigo como proyecto, limpiamos.- 

tareas = tareas.drop_duplicates(subset=["CODCLI2","PROYECTO"], keep="first")
tareas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1101 entries, 0 to 1750
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PROYECTO      1100 non-null   object 
 1   PERIODICIDAD  928 non-null    object 
 2   ORDEN         1037 non-null   float64
 3   TIPO          1100 non-null   object 
 4   CODCLI2       1082 non-null   Int64  
dtypes: Int64(1), float64(1), object(3)
memory usage: 52.7+ KB


In [22]:
# Union pero manteniendo tambien los que no tienen proyecto para poder visibilizarlo en el mapa: 

Tabla_Proyectos_Clientes = pd.merge(ubicaciones_clientes,
                                    tareas[['CODCLI2', 'PROYECTO', "PERIODICIDAD", "ORDEN", "TIPO"]],
                                    on='CODCLI2',
                                    how='left')

Tabla_Proyectos_Clientes['PROYECTO'].fillna('sin proyecto', inplace=True)


Tabla_Proyectos_Clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1210 entries, 0 to 1209
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   CODCLI2       1210 non-null   int64  
 1   NOMCLI        1207 non-null   object 
 2   LATITUD       1210 non-null   float64
 3   LONGITUD      1210 non-null   float64
 4   UBICACIÓN     1210 non-null   object 
 5   METODO        1210 non-null   object 
 6   CONFIANZA     1210 non-null   int64  
 7   PROYECTO      1210 non-null   object 
 8   PERIODICIDAD  796 non-null    object 
 9   ORDEN         899 non-null    float64
 10  TIPO          943 non-null    object 
dtypes: float64(3), int64(2), object(6)
memory usage: 104.1+ KB


In [23]:
Tabla_Proyectos_Clientes.head()

,CODCLI2,NOMCLI,LATITUD,LONGITUD,UBICACIÓN,METODO,CONFIANZA,PROYECTO,PERIODICIDAD,ORDEN,TIPO
0,82122,NUEVOS RUMBOS SALINAS,-34.765666,-55.837849,"http://maps.google.com/?q=-34.7656659,-55.8378493",google_q_param,90,Ruta Hugo Martes,CADA MARTES,13.0,Ruta
1,82122,NUEVOS RUMBOS SALINAS,-34.765666,-55.837849,"http://maps.google.com/?q=-34.7656659,-55.8378493",google_q_param,90,Ruta Hugo Jueves,CADA JUEVES,12.0,Ruta
2,82122,NUEVOS RUMBOS SALINAS,-34.765666,-55.837849,"http://maps.google.com/?q=-34.7656659,-55.8378493",google_q_param,90,Ruta Hugo Sabado,CADA sab,19.0,Ruta
3,82122,NUEVOS RUMBOS SALINAS,-34.765666,-55.837849,"http://maps.google.com/?q=-34.7656659,-55.8378493",google_q_param,90,💲Creditos Hugo,NaN,53.0,💲Cre
4,81125,LA ESQUINA jacqueline vanesca (cobranza),-34.780645,-55.838592,"http://maps.google.com/?q=-34.78064530920805,-...",google_q_param,90,Ruta Hugo Martes,CADA MARTES,30.0,Ruta


In [24]:
# Ordeno como estaban: 

Tabla_Proyectos_Clientes = Tabla_Proyectos_Clientes[['CODCLI2', 'NOMCLI', "PROYECTO", "PERIODICIDAD", "ORDEN", "TIPO","UBICACIÓN"]]
Tabla_Proyectos_Clientes.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1210 entries, 0 to 1209
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   CODCLI2       1210 non-null   int64  
 1   NOMCLI        1207 non-null   object 
 2   PROYECTO      1210 non-null   object 
 3   PERIODICIDAD  796 non-null    object 
 4   ORDEN         899 non-null    float64
 5   TIPO          943 non-null    object 
 6   UBICACIÓN     1210 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 66.3+ KB


In [25]:
Tabla_Proyectos_Clientes.sample(30)

,CODCLI2,NOMCLI,PROYECTO,PERIODICIDAD,ORDEN,TIPO,UBICACIÓN
695,80116,ALMACEN EL HORNERO,Ruta Dario Viernes,CADA VIERNES,48.0,Ruta,"http://maps.google.com/?q=-34.76127994910307,-..."
703,85029,EL ENCUENTRO. Tabaré,sin proyecto,NaN,NaN,NaN,"http://maps.google.com/?q=-34.7634123855835,-5..."
24,62619,"62619, LETICIA COSTA",sin proyecto,NaN,NaN,NaN,https://maps.app.goo.gl/JhiEkkF7XM52CKQy8
146,81233,"81233,EL LOLO",💲Creditos Administracion,NaN,6.0,💲Cre,Marcador 26\n2024/10/18 @ 17:14:54\nhttp://map...
265,80290,Parador Toto,Ruta Dario Miercoles,CADA MIERCOLES,14.0,Ruta,"http://maps.google.com/?q=-34.771252, -55.756427"
107,12289,"12289, ALOBA SRL (pasa pedido)",Clientes Especiales,NaN,164.0,Clie,Marcador 4\n2024/10/17 @ 08:38:40\nhttp://maps...
1023,13075,LA BUENA SUERTE,sin proyecto,NaN,NaN,NaN,"http://maps.google.com/?q=-34.76751955311376,-..."
614,80146,PAPELERIA NICO,Ruta Alejandro Lunes,CADA LUNES,33.0,Ruta,"http://maps.google.com/?q=-34.766108563600454,..."
147,82210,"82210,PABLO PORTUARIOS",Ruta Hugo Miercoles,CADA MIERCOLES,21.0,Ruta,Marcador 27\n2024/10/18 @ 17:17:38\nhttp://map...
779,22878,LA BARRA,sin proyecto,NaN,NaN,NaN,"http://maps.google.com/?q=-34.763207737972365,..."


## Exportamos 


In [26]:
# Exportamos 
Tabla_Proyectos_Clientes.to_excel('../Limpia/Tabla_Proyectos_Clientes.xlsx', index=False)